In [1]:
import polars as pl
import pickle

AXIELL_MARC_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-marc-works.parquet"
AXIELL_SOURCE_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-source-works.parquet"

def from_parquet_snapshot(path: str):
    df = pl.read_parquet(path)
    return {row["id"]: pickle.loads(row["body"]) for row in df.iter_rows(named=True)}

def save_parquet_snapshot(data: dict, path: str):
    df = pl.DataFrame({
        "id": list(data.keys()),
        "body": [pickle.dumps(v) for v in data.values()],
    })
    df.write_parquet(path)

axiell_marc_works = from_parquet_snapshot(AXIELL_MARC_WORKS_PATH)
print(len(axiell_marc_works))

216939


In [2]:
from adapters.transformers.builders.axiell_work_builder import AxiellWorkBuilder
from datetime import datetime

transformed_axiell_works = {}
missing_id_field = []
missing_ref_no = []
missing_level = []

for i, (work_id, record) in enumerate(axiell_marc_works.items()):
    try:
        axiell_work = AxiellWorkBuilder(record=record, last_modified=datetime.now()).transform_work()
        transformed_axiell_works[work_id] = axiell_work
    except Exception as e:
        if "Missing RefNo" in str(e):
            missing_ref_no.append(work_id)
        elif "Empty id field" in str(e):
            missing_id_field.append(work_id)
        elif "Missing hierarchical level (work type) on record" in str(e):
            missing_level.append(work_id)        
        elif "Invalid century" in str(e):
            print("Invalid century")
        else:
            print(work_id, record)
            raise e

    if i % 20000 == 0:
        print(i)

print("Successfully transformed works:", len(transformed_axiell_works))
print("Axiell works with missing ID field:", len(missing_id_field))
print("Axiell works with a missing CALM RefNo:", len(missing_ref_no))
print("Axiell works with a missing work type", len(missing_level))

0
2026-07-02 09:20:27 [warning  ] Unclear how to create a 'terms of use' note record_id=09a8f1d2-1347-44f9-8541-76d828161cd8
2026-07-02 09:20:27 [warning  ] Unrecognised Axiell access status value record_id=7f6a9592-587a-4b0e-9792-795ed34310c4 status=DATAISSUES
2026-07-02 09:20:27 [warning  ] Unrecognised Axiell access status value record_id=7f6a9592-587a-4b0e-9792-795ed34310c4 status=DATAISSUES
2026-07-02 09:20:27 [warning  ] Unclear how to create a 'terms of use' note record_id=8345b4bb-2fe3-4c41-bde7-bd6a13e39bbe
2026-07-02 09:20:28 [warning  ] Unclear how to create a 'terms of use' note record_id=d3e17ed9-1f15-4e14-ad01-bbeef3bed71e
2026-07-02 09:20:28 [warning  ] Unclear how to create a 'terms of use' note record_id=f8c983be-c19d-4df6-b78c-3136343aca19
2026-07-02 09:20:28 [warning  ] Unclear how to create a 'terms of use' note record_id=a8d0bb48-11c6-47e1-9283-239171c7e0fe
2026-07-02 09:20:28 [warning  ] Unclear how to create a 'terms of use' note record_id=cd3fca1e-b09e-4da7-999d

In [3]:
save_parquet_snapshot(transformed_axiell_works, AXIELL_SOURCE_WORKS_PATH)

In [6]:
len(transformed_axiell_works)

211417